<a href="https://colab.research.google.com/github/mybenkhadda/GEMS---drivendata/blob/main/notebooks/colab/00_setup_and_drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 · Setup: Drive, package, GPU, paths, manifests

**GitHub = code · Colab `/content` = fast ephemeral execution · Google Drive (`MyDrive/GEMS_2`) = persistent data and artifacts.**

Run this notebook at the start of every Colab session (or copy its first cell into any other notebook: every notebook here starts with the same idempotent bootstrap).

1. Mount Drive · 2. clone the repository at a pinned commit and `pip install -e .[colab]` · 3. check PyTorch / GPU · 4. record package version + commit · 5. free disk / VRAM · 6. validate data paths · 7. create the workspace folders · 8. stage the core data from Drive to `/content/gems_work` · 9. reload experiment manifests · 10. environment report.

Nothing here trains a model. No credentials are stored in the notebook: a private repository is cloned with a token read from Colab *Secrets* (`GITHUB_TOKEN`).

In [1]:
# --- session bootstrap (idempotent; identical in notebooks 00-07) ---------------------------------
REPO = 'mybenkhadda/GEMS---drivendata'          # GitHub owner/name of this project
COMMIT = 'main'                    # pin a commit SHA for reproducible experiments
DRIVE_ROOT = '/content/drive/MyDrive/GEMS_2'
import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
url = f'https://github.com/{REPO}.git'   # token-free remote: a token never goes into the URL / .git/config
# private fork only: GITHUB_TOKEN (Colab Secret) is sent as an HTTP header through env vars, per command
genv = {**os.environ, 'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader', 'GIT_CONFIG_VALUE_0': 'AUTHORIZATION: basic ' + __import__('base64').b64encode(f'x-access-token:{token}'.encode()).decode()} if token else None
if not os.path.exists('/content/GEMS/.git'):
    subprocess.run(['git', 'clone', '-q', url, '/content/GEMS'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'remote', 'set-url', 'origin', url], check=True)  # drop an old embedded token
subprocess.run(['git', '-C', '/content/GEMS', 'fetch', '-q', 'origin'], check=True, env=genv)
subprocess.run(['git', '-C', '/content/GEMS', 'checkout', '-q', COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/GEMS[colab]'], check=True)
if '/content/GEMS' not in sys.path:
    sys.path.insert(0, '/content/GEMS')       # editable install is not visible to an already-running kernel
from gems_bench.runtime import colab
colab.configure(drive_root=DRIVE_ROOT)
print(subprocess.run(['git', '-C', '/content/GEMS', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout)

Mounted at /content/drive
32c0ad518fc375fdd872424a2a78503d54def93f



## Download and organize the data (once)

DrivenData links are **personal and expire**. Log in, open the competition's *Data download* page, copy each file's link and paste it into `DATA` below. If the data comes as one zip, paste its link into `ARCHIVE_URL` instead.

Each file is fetched with `wget -c` (resumable) to the fast local disk. It is then checked (it must open as a GeoTIFF, and its size is compared with the known release), moved into the workspace layout (`data/raw/geodawn`, `data/raw/faults`) and copied to Drive with a checksum.

Files already on Drive are skipped, so re-running this cell is harmless. Never commit or share the links.

The second cell optionally lists the USGS 1 m DEM products for the study area with `wget` (metadata only; no tiles).

The legacy Phase 1–4 `outputs/` (feature caches, folds, model maps) are not downloadable: upload them once from the local project (`docs/GOOGLE_DRIVE_LAYOUT.md`).

In [2]:

# ============================================================
# GEMS 2.0 — One-time persistent data collection
# DrivenData GeoDAWN + labels + submission + USGS 1m DEM
#
# Download once -> verify -> Google Drive -> reuse forever
#
# Requires: requests, rasterio
# ============================================================

import os
import json
import time
import shutil
import hashlib
import logging
from pathlib import Path
from urllib.parse import urlsplit, unquote
from datetime import datetime, timezone

import requests
import rasterio
import pandas as pd

# ------------------------------------------------------------
# 1. CONFIGURATION
# ------------------------------------------------------------

LOCAL = Path(colab.LOCAL_ROOT).resolve()
DRIVE = Path(DRIVE_ROOT).resolve()

# Full download can be large. Set to an integer for a pilot.
MAX_DEM_TILES = None  # None = download all DEM URLs
DOWNLOAD_DEM = True

# Re-hash existing Drive files on every run (slower but stricter)
VERIFY_EXISTING_SHA = False

CHUNK_BYTES = 8 * 1024 * 1024
MAX_RETRIES = 5

DATA_ROOT = Path("data/raw")
DEM_ROOT = DATA_ROOT / "dem_1m"
MANIFEST_PATH = DRIVE / "data/manifests/download_manifest.json"

# ------------------------------------------------------------
# 2. OFFICIAL COMPETITION FILES
# ------------------------------------------------------------

def _url(secret):
    """Signed DrivenData link from Colab Secrets (key icon). Links expire: never paste them into a cell."""
    try:
        from google.colab import userdata
        return userdata.get(secret) or f"MISSING-COLAB-SECRET-{secret}"
    except Exception:                 # only used when the file is absent on Drive
        return f"MISSING-COLAB-SECRET-{secret}"

DATA = {
    "data/raw/geodawn/gems-geodawn-numerical-features.tif": (
        _url("GEMS_URL_FEATURES"),
        418_912_844,
    ),
    "data/raw/faults/existing_faults.tif": (
        _url("GEMS_URL_FAULTS"),
        425_830,
    ),
    "data/raw/example_submission.tif": (
        _url("GEMS_URL_SUBMISSION"),
        1_599_597,
    ),
}

TNM_JSON_URL = _url("GEMS_URL_DEM_JSON")

JSON_REL = DATA_ROOT / "tnm_items.json"

# ------------------------------------------------------------
# 3. LOGGING + MANIFEST
# ------------------------------------------------------------

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    force=True,
)
log = logging.getLogger("GEMS_DOWNLOAD")

LOCAL.mkdir(parents=True, exist_ok=True)
DRIVE.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)

if MANIFEST_PATH.exists():
    try:
        manifest = json.loads(MANIFEST_PATH.read_text())
    except (ValueError, OSError):
        raise RuntimeError(
            "Download manifest is unreadable. "
            "Inspect it before continuing."
        )
else:
    manifest = {"version": 1, "files": {}}


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(CHUNK_BYTES), b""):
            digest.update(chunk)
    return digest.hexdigest()


def save_manifest():
    temp = MANIFEST_PATH.with_name(MANIFEST_PATH.name + ".tmp")
    temp.write_text(json.dumps(manifest, indent=2))
    temp.replace(MANIFEST_PATH)


def validate_file(path, expected=None, is_tif=True):
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError(f"Missing or empty file: {path}")

    size = path.stat().st_size

    if expected is not None and size != expected:
        log.warning(
            "%s: size %s differs from reference %s",
            path.name, size, expected,
        )

    if is_tif:
        with rasterio.open(path) as src:
            if src.count <= 0 or src.width <= 0 or src.height <= 0:
                raise ValueError(f"Invalid raster dimensions: {path}")

            # Read a small window to detect some corrupt TIFFs.
            src.read(
                1,
                window=((0, min(1, src.height)),
                        (0, min(1, src.width))),
            )

    else:
        with open(path, "r", encoding="utf-8") as f:
            json.load(f)

    return size


# ------------------------------------------------------------
# 4. RESUMABLE HTTP DOWNLOAD
# ------------------------------------------------------------

session = requests.Session()
session.headers.update({"User-Agent": "GEMS-Research-Downloader/1.0"})


def download_resumable(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".partial")

    for attempt in range(1, MAX_RETRIES + 1):
        offset = partial.stat().st_size if partial.exists() else 0

        headers = {"Range": f"bytes={offset}-"} if offset else {}

        try:
            with session.get(
                url,
                headers=headers,
                stream=True,
                timeout=(30, 120),
                allow_redirects=True,
            ) as response:

                if response.status_code in (401, 403):
                    raise PermissionError(
                        f"Access denied for {destination.name}. "
                        "A signed DrivenData link may have expired."
                    )

                if response.status_code == 416 and offset:
                    # The existing partial may already contain the
                    # whole remote file. Validate it below.
                    response.close()
                    if destination.suffix.lower() in (".tif", ".tiff"):
                        validate_file(partial, is_tif=True)
                    else:
                        validate_file(partial, is_tif=False)
                    partial.replace(destination)
                    return destination

                response.raise_for_status()

                # Server ignored Range: restart instead of appending.
                mode = "ab"
                if offset == 0 or response.status_code == 200:
                    mode = "wb"

                if offset and response.status_code == 206:
                    content_range = response.headers.get("Content-Range", "")
                    if not content_range.startswith(f"bytes {offset}-"):
                        raise IOError(
                            f"Unexpected Content-Range: {content_range}"
                        )

                with open(partial, mode) as f:
                    for chunk in response.iter_content(
                        chunk_size=CHUNK_BYTES
                    ):
                        if chunk:
                            f.write(chunk)

            # Verify against Content-Length / Content-Range where possible.
            if response.status_code == 206:
                content_range = response.headers.get("Content-Range", "")
                total_text = content_range.rsplit("/", 1)[-1]
                remote_total = (
                    int(total_text) if total_text.isdigit() else None
                )
            else:
                length = response.headers.get("Content-Length")
                remote_total = int(length) if length else None

            if remote_total is not None:
                actual = partial.stat().st_size
                if actual != remote_total:
                    raise IOError(
                        f"Incomplete download: {actual}/{remote_total} bytes"
                    )

            if destination.suffix.lower() in (".tif", ".tiff"):
                validate_file(partial, is_tif=True)
            else:
                validate_file(partial, is_tif=False)

            partial.replace(destination)
            return destination

        except PermissionError:
            raise

        except Exception as exc:
            log.warning(
                "Attempt %d/%d failed for %s: %s",
                attempt, MAX_RETRIES, destination.name, exc,
            )
            if attempt == MAX_RETRIES:
                raise
            time.sleep(min(2 ** attempt, 30))

    raise RuntimeError(f"Download failed: {destination.name}")


# ------------------------------------------------------------
# 5. VERIFY + PERSIST TO GOOGLE DRIVE
# ------------------------------------------------------------

def collect_file(relative_path, url, expected=None, is_tif=True):
    rel = Path(relative_path)
    local_path = LOCAL / rel
    drive_path = DRIVE / rel

    key = rel.as_posix()
    previous = manifest["files"].get(key, {})

    # Drive is the persistent source of truth.
    if drive_path.exists():
        try:
            size = validate_file(drive_path, expected, is_tif)

            if VERIFY_EXISTING_SHA and previous.get("sha256"):
                digest = sha256_file(drive_path)
                if digest != previous["sha256"]:
                    raise ValueError("Existing Drive checksum mismatch")

            log.info("ON DRIVE | %s (%0.2f MB)", key, size / 1e6)

            if key not in manifest["files"]:
                manifest["files"][key] = {
                    "size_bytes": size,
                    "sha256": sha256_file(drive_path),
                    "verified_at": utc_now(),
                    "status": "ready",
                }
                save_manifest()

            return drive_path

        except Exception as exc:
            log.warning("Invalid Drive copy: %s: %s", key, exc)
            # Do not overwrite a questionable permanent copy silently.
            raise RuntimeError(
                f"Please inspect invalid Drive artifact: {drive_path}"
            ) from exc

    # Download into Colab's local working directory.
    if local_path.exists():
        try:
            validate_file(local_path, expected, is_tif)
            log.info("LOCAL CACHE | %s", key)
        except Exception:
            local_path.unlink()

    if not local_path.exists():
        log.info("DOWNLOADING | %s", key)
        download_resumable(url, local_path)

    size = validate_file(local_path, expected, is_tif)
    digest = sha256_file(local_path)

    # Copy to a temporary Drive file, then verify and rename.
    drive_path.parent.mkdir(parents=True, exist_ok=True)

    drive_partial = drive_path.with_name(drive_path.name + ".partial")

    if drive_partial.exists():
        drive_partial.unlink()

    shutil.copyfile(local_path, drive_partial)

    if (
        drive_partial.stat().st_size != size
        or sha256_file(drive_partial) != digest
    ):
        raise IOError(f"Drive transfer verification failed: {key}")

    drive_partial.replace(drive_path)

    manifest["files"][key] = {
        "size_bytes": size,
        "sha256": digest,
        "verified_at": utc_now(),
        "status": "ready",
    }
    save_manifest()

    log.info("SAVED TO DRIVE | %s (%0.2f MB)", key, size / 1e6)

    # Local copy is disposable after verified persistence.
    local_path.unlink(missing_ok=True)

    return drive_path


# ------------------------------------------------------------
# 6. COLLECT OFFICIAL DATA
# ------------------------------------------------------------

for rel, (url, expected) in DATA.items():
    collect_file(
        relative_path=rel,
        url=url,
        expected=expected,
        is_tif=True,
    )


# ------------------------------------------------------------
# 7. COLLECT USGS DEM URL JSON
# ------------------------------------------------------------

json_path = collect_file(
    relative_path=JSON_REL,
    url=TNM_JSON_URL,
    expected=None,
    is_tif=False,
)

with open(json_path, "r", encoding="utf-8") as f:
    tnm_data = json.load(f)

# Accept the list-of-URLs format shown in the competition.
if isinstance(tnm_data, list):
    dem_urls = [
        item if isinstance(item, str) else item.get("url")
        for item in tnm_data
        if isinstance(item, (str, dict))
    ]
elif isinstance(tnm_data, dict):
    items = tnm_data.get("items", tnm_data.get("urls", []))
    dem_urls = [
        item if isinstance(item, str) else item.get("url")
        for item in items
        if isinstance(item, (str, dict))
    ]
else:
    raise ValueError("Unsupported tnm_items.json structure")

dem_urls = list(dict.fromkeys(
    url for url in dem_urls
    if isinstance(url, str) and url.startswith("https://")
))

log.info("DEM URLs found: %d", len(dem_urls))


# ------------------------------------------------------------
# 8. PRESERVE PROJECT IDENTITY FOR OVERLAPPING DEM TILES
# ------------------------------------------------------------

def dem_relative_path(url):
    path = unquote(urlsplit(url).path)
    parts = Path(path).parts

    if "Projects" not in parts:
        raise ValueError(f"Unknown USGS tile URL structure: {url}")

    project_idx = parts.index("Projects")
    project = parts[project_idx + 1]
    filename = parts[-1]

    if not filename.lower().endswith((".tif", ".tiff")):
        raise ValueError(f"Not a GeoTIFF: {filename}")

    if project in ("", ".", "..") or "/" in project or "\\" in project:
        raise ValueError(f"Unsafe DEM project name: {project}")

    if filename in ("", ".", ".."):
        raise ValueError("Unsafe DEM filename")

    return DEM_ROOT / project / filename


dem_records = []
seen_paths = {}

for url in dem_urls:
    rel = dem_relative_path(url)
    key = rel.as_posix()

    if key in seen_paths and seen_paths[key] != url:
        raise ValueError(f"Different URLs map to the same path: {key}")

    seen_paths[key] = url
    dem_records.append({
        "project": rel.parent.name,
        "filename": rel.name,
        "relative_path": key,
        "url": url,
    })

dem_inventory = pd.DataFrame(dem_records)

inventory_path = DRIVE / "data/manifests/dem_url_inventory.csv"
inventory_path.parent.mkdir(parents=True, exist_ok=True)

dem_inventory.to_csv(inventory_path, index=False)

print("\nUSGS DEM projects:")
print(dem_inventory.groupby("project").size().to_string())

print("\nTotal DEM tiles:", len(dem_inventory))
print("Inventory:", inventory_path)


# ------------------------------------------------------------
# 9. ONE-TIME DEM DOWNLOADS
# ------------------------------------------------------------

if DOWNLOAD_DEM:
    selected = dem_records
    if MAX_DEM_TILES is not None:
        selected = selected[:MAX_DEM_TILES]

    successes = 0
    failures = []

    for i, item in enumerate(selected, start=1):
        print(f"\nDEM {i}/{len(selected)}: {item['filename']}")

        try:
            collect_file(
                relative_path=item["relative_path"],
                url=item["url"],
                expected=None,
                is_tif=True,
            )
            successes += 1
        except Exception as exc:
            failures.append({
                "relative_path": item["relative_path"],
                "error": str(exc),
            })
            log.error("FAILED | %s | %s", item["filename"], exc)

    print("\nDEM downloaded/reused:", successes)
    print("DEM failures:", len(failures))

    if failures:
        failure_path = DRIVE / "data/manifests/dem_failures.csv"
        pd.DataFrame(failures).to_csv(failure_path, index=False)
        print("Failure report:", failure_path)

else:
    print("DEM download disabled; inventory saved only.")


# ------------------------------------------------------------
# 10. SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame([
    {
        "path": key,
        "size_gb": round(item["size_bytes"] / 1024**3, 4),
        "status": item["status"],
        "sha256": item["sha256"],
    }
    for key, item in manifest["files"].items()
])

summary_path = DRIVE / "data/manifests/download_summary.csv"
summary.to_csv(summary_path, index=False)

print("\nDATA COLLECTION COMPLETE")
print("Drive root:", DRIVE)
print("Total cached files:", len(summary))
print("Total cached GB:", round(summary["size_gb"].sum(), 2))
print("Manifest:", MANIFEST_PATH)


2026-10-21 23:40:05,835 | INFO | ON DRIVE | data/raw/geodawn/gems-geodawn-numerical-features.tif (418.91 MB)
2026-10-21 23:40:06,434 | INFO | ON DRIVE | data/raw/faults/existing_faults.tif (0.43 MB)
2026-10-21 23:40:06,896 | INFO | ON DRIVE | data/raw/example_submission.tif (1.60 MB)
2026-10-21 23:40:07,357 | INFO | ON DRIVE | data/raw/tnm_items.json (0.15 MB)
2026-10-21 23:40:07,367 | INFO | DEM URLs found: 894



USGS DEM projects:
project
CA_SierraNevada_B22                  22
NV_EastCentral_2021_D21             141
NV_Humboldt_2021_D21                 15
NV_NorthWestElko_2020_D20            73
NV_Reno_Carson_QL1_2017              16
NV_Reno_Carson_QL2_2017              34
NV_UpperHumboldt_2016                 3
NV_WestCentral_EarthMRI_2020_D20    590

Total DEM tiles: 894
Inventory: /content/drive/MyDrive/GEMS_2/data/manifests/dem_url_inventory.csv

DEM 1/894: USGS_1M_10_x75y441_CA_SierraNevada_B22.tif


2026-10-21 23:40:10,331 | INFO | ON DRIVE | data/raw/dem_1m/CA_SierraNevada_B22/USGS_1M_10_x75y441_CA_SierraNevada_B22.tif (238.66 MB)



DEM 2/894: USGS_1M_10_x75y441_NV_WestCentral_EarthMRI_2020_D20.tif


2026-10-21 23:40:11,792 | INFO | ON DRIVE | data/raw/dem_1m/NV_WestCentral_EarthMRI_2020_D20/USGS_1M_10_x75y441_NV_WestCentral_EarthMRI_2020_D20.tif (189.38 MB)


...


DEM 893/894: USGS_one_meter_x76y441_NV_Reno_Carson_QL2_2017.tif


2026-10-22 00:12:47,384 | INFO | SAVED TO DRIVE | data/raw/dem_1m/NV_Reno_Carson_QL2_2017/USGS_one_meter_x76y441_NV_Reno_Carson_QL2_2017.tif (447.02 MB)
2026-10-22 00:12:47,464 | INFO | DOWNLOADING | data/raw/dem_1m/NV_Reno_Carson_QL2_2017/USGS_one_meter_x76y442_NV_Reno_Carson_QL2_2017.tif



DEM 894/894: USGS_one_meter_x76y442_NV_Reno_Carson_QL2_2017.tif


2026-10-22 00:13:04,022 | INFO | SAVED TO DRIVE | data/raw/dem_1m/NV_Reno_Carson_QL2_2017/USGS_one_meter_x76y442_NV_Reno_Carson_QL2_2017.tif (381.78 MB)



DEM downloaded/reused: 140
DEM failures: 754
Failure report: /content/drive/MyDrive/GEMS_2/data/manifests/dem_failures.csv

DATA COLLECTION COMPLETE
Drive root: /content/drive/MyDrive/GEMS_2
Total cached files: 380
Total cached GB: 82.48
Manifest: /content/drive/MyDrive/GEMS_2/data/manifests/download_manifest.json


In [3]:
# Optional: USGS TNM 1 m DEM URL list for the study area via the TNM Access API (metadata only: NO tiles here;
# tiles are downloaded later by notebook 05 after explicit approval). Skip if you already have your own JSON.
FETCH_DEM_URL_LIST = False
dem_json = Path(DRIVE_ROOT) / 'data/raw/dem_1m/tnm_urls.json'
if FETCH_DEM_URL_LIST and not dem_json.exists():
    import json, urllib.parse
    import rasterio
    from rasterio.warp import transform_bounds
    with rasterio.open(LOCAL / 'data/raw/geodawn/gems-geodawn-numerical-features.tif') as src:
        w, s, e, n = transform_bounds(src.crs, 'EPSG:4326', *src.bounds)
    items, offset = [], 0
    while True:
        q = urllib.parse.urlencode({'datasets': 'Digital Elevation Model (DEM) 1 meter', 'prodFormats': 'GeoTIFF',
                                    'bbox': f'{w:.4f},{s:.4f},{e:.4f},{n:.4f}', 'max': 500, 'offset': offset})
        page = LOCAL / f'data/raw/dem_1m/_tnm_page_{offset}.json'
        page.unlink(missing_ok=True)                  # a page must never be "continued" by wget -c
        wget('https://tnmaccess.nationalmap.gov/api/v1/products?' + q, page)
        r = json.loads(page.read_text())
        items += r.get('items', [])
        offset += 500
        if offset >= int(r.get('total', 0)) or not r.get('items'):
            break
    out = LOCAL / 'data/raw/dem_1m/tnm_urls.json'
    out.write_text(json.dumps({'items': items, 'bbox_wgs84': [w, s, e, n]}, indent=1))
    for p in out.parent.glob('_tnm_page_*.json'):
        p.unlink()
    gib = sum(i.get('sizeInBytes') or 0 for i in items) / 2**30
    print(f'{len(items)} products listed ({gib:.1f} GiB if ALL were downloaded - the pilot uses ~24 tiles)')
    colab.sync().push(['data/raw/dem_1m/tnm_urls.json'])
elif dem_json.exists():
    print('DEM URL list on Drive:', dem_json)

In [4]:
# PyTorch / GPU and the auto-configuration the trainers will use
from gems_bench.training import gpu
g = gpu.detect()
print(g)
print('autoconfig (256 px, effective batch 16, 29 channels):', gpu.autoconfig(g, patch_px=256, effective_batch=16, n_channels=29))

{'cuda': False, 'name': None, 'vram_gb': 0.0, 'capability': None, 'count': 0}
autoconfig (256 px, effective batch 16, 29 channels): {'amp': False, 'data_on_gpu': False, 'batch_size': 4, 'grad_accum': 4, 'infer_batch': 4, 'cudnn_benchmark': False}


In [5]:
# Drive layout: create missing workspace folders on Drive and locally (never deletes anything)
from pathlib import Path
from gems_bench.paths import WORKSPACE_DIRS
for d in WORKSPACE_DIRS:
    (Path(DRIVE_ROOT) / d).mkdir(parents=True, exist_ok=True)
    (Path(colab.LOCAL_ROOT) / d).mkdir(parents=True, exist_ok=True)
required = ['data/raw/geodawn/gems-geodawn-numerical-features.tif', 'data/raw/faults/existing_faults.tif',
            'data/raw/geodawn/example_submission.tif', 'outputs/benchmark/folds', 'outputs/phase4/experiments']
for r in required:
    print(('OK      ' if (Path(DRIVE_ROOT) / r).exists() else 'MISSING ') + r)
print('See docs/GOOGLE_DRIVE_LAYOUT.md for what to upload once from the local project.')

OK      data/raw/geodawn/gems-geodawn-numerical-features.tif
OK      data/raw/faults/existing_faults.tif
MISSING data/raw/geodawn/example_submission.tif
MISSING outputs/benchmark/folds
MISSING outputs/phase4/experiments
See docs/GOOGLE_DRIVE_LAYOUT.md for what to upload once from the local project.


In [6]:
# Stage the core data to the fast local disk (unchanged files are skipped on later sessions)
staged = colab.stage_core()
staged.groupby('action').bytes.agg(['count', 'sum'])

[stage] not on Drive (see docs/GOOGLE_DRIVE_LAYOUT.md): ['outputs/benchmark/folds', 'outputs/benchmark/datasets', 'outputs/eda/manifest.json', 'outputs/eda/datasets']
[sync in] 2 copied (399.9 MiB), 0 unchanged, 11s: ['data/raw/geodawn', 'data/raw/faults', 'experiments', 'model_registry']


,count,sum
action,,
copy,2,419338674


In [7]:
# Existing experiment manifests and the model registry (from Drive)
import json
import pandas as pd
from gems_bench.artifacts import ModelRegistry
rows = []
for m in sorted(Path(DRIVE_ROOT, 'experiments').glob('*/manifest.json')):
    j = json.loads(m.read_text())
    rows.append({'experiment': j['experiment_id'], 'kind': j['kind'], 'commit': j.get('git_commit', '')[:10],
                 **{s: v['status'] for s, v in j['stages'].items()}})
display(pd.DataFrame(rows))
display(ModelRegistry(Path(DRIVE_ROOT) / 'model_registry').table())

""


,name,status,kind,experiment_id,oof,discovery,config_hash,git_commit,geo_dti,disc_dti_excl,disc_dti_retained_fp,protocol_hash,updated,notes


In [8]:
colab.environment_report()

,key,value
0,package,gems-fault-discovery
1,version,0.5.0
2,git_commit,32c0ad518fc375fdd872424a2a78503d54def93f
3,python,3.13.16
4,gpu,None
5,vram_gb,0.0
6,capability,None
7,torch,2.11.0+cpu
8,cuda,None
9,free_gb:/content,173.9
